DEPENDÊNCIAS DO SISTEMA OPERACIONAL

- `Python 3.11.5 (recomendado >= 3.8, < 3.12)`
- `PostgreSQL 12 (>= 9.4, < 14)`


LINK DO TUTORIAL:
- https://docs.taiga.io/setup-production.html#setup-prod-from-source-code

#### Passo 1: **Inicialização**

É necessário efetuar a instalação do Docker Engine no Ubuntu:

*[Tutorial Instalação do Docker Engine](https://docs.docker.com/engine/install/ubuntu/)*

Após isso, insira os comandos abaixo para instalar e atualizar o sistema com um conjunto de ferramentas essenciais de desenvolvimento no Ubuntu. 


In [ ]:
# INÍCIO
sudo apt-get update
sudo apt-get upgrade
sudo apt-get install -y build-essential binutils-doc autoconf flex bison libjpeg-dev
sudo apt-get install -y libfreetype6-dev zlib1g-dev libzmq3-dev libgdbm-dev libncurses5-dev
sudo apt-get install -y automake libtool curl git tmux gettext

#### Passo 2: **Instalação do Python e Node.js**

É obrigatório a instalação do Python 3.

Insira os comandos abaixo para instalar o *Python* e o *Node.js*. 

In [ ]:
# PYTHON 3 
sudo apt-get install -y python3 python3-pip python3-dev python3-venv #máquina virtual
sudo apt-get install -y libxml2-dev libxslt-dev
sudo apt-get install -y libssl-dev libffi-dev

sudo apt install python3-distutils #no Ubuntu 25.04 use o python3-distutils-extra

# NODE JS
curl -sL https://deb.nodesource.com/setup_16.x | sudo -E bash - #instalar via NVM
sudo apt-get install -y nodejs

#### Passo 2.1: **Instalação do Node.js via NVM**

O NVM (Node Version Manager) é um gerenciador de versões do Node.js.
Ele serve para instalar, trocar e gerenciar várias versões do Node.js na mesma máquina.

Para a instalação do NVM, basta seguir o tutorial: [Acesse o tutorial](https://github.com/nvm-sh/nvm?tab=readme-ov-file#installing-and-updating)


Exemplo para instalar e utilizar:   
```
nvm install 16
nvm use 16
```

In [ ]:
# TEM ESSA OPÇÃO NO TUTORIAL, MAS LOCALMENTE NÃO EXECUTEI
sudo adduser taiga
sudo adduser taiga sudo
sudo su taiga
cd ~

#### Passo 3: **Dependências do Backend**

Existem duas formas de criar ambientes isolados para Python, mas cada uma funciona de modo diferente.
- venv (evita conflitos de versões entre projetos)
- Anaconda (distribuição completa de Python)

Além disso, é necessário instalar o taiga protected.


In [ ]:
### CRIAR UMA MÁQUINA VIRTUAL / USEI O ANACONDA-NAVIGATOR
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip setuptools wheel
python -m pip install --upgrade pip-tools

pip install -r requirements.txt 
pip install -r requirements-devel.txt
pip install -r requirements-tests.txt


In [ ]:
pip install git+https://@github.com/celtab/apps-tadt/taigaio/taiga-contrib-protected.git@stable#egg=taiga-contrib-protected

# Ao rodar o comando acima, observe que o vscode vai solicitar seu usuário e senha
# pois a branch está no nosso repositório privado celtab/apps-tadt/taigaio/...

Edit config.py and configure:

* Taiga secret key: **it’s important** to change it. It must have the same value as the secret key in `taiga-events` and `taiga-protected`
* Taiga urls: configure where Taiga would be served using `TAIGA_URL`, `SITES` and `FORCE_SCRIPT_NAME` (see examples below)
* Connection to PostgreSQL; check `DATABASES` section in the file
* Connection to RabbitMQ for `taiga-events`; check "EVENTS" section in the file
* Connection to RabbitMQ for `taiga-async`; check "TAIGA ASYNC" section in the file
* Credentials for email; check "EMAIL" section in the file
* Enable/disable anonymous telemetry; check "TELEMETRY" section in the file

In [ ]:
# Example to configure Taiga in subdomain:

TAIGA_SCHEME = "https"
TAIGA_DOMAIN = "taiga.mycompany.com"
FORCE_SCRIPT_NAME = ""

# Example to configure Taiga in subpath:
TAIGA_SCHEME = "https"
TAIGA_DOMAIN = "taiga.mycompany.com"
FORCE_SCRIPT_NAME = "/taiga"

#### Passo 3.1: **Caso você utilize Anaconda**
Insira os comandos abaixo. Senão, apenas desconsidere esta parte, pois todos estes comandos estão dentro do venv. 

In [ ]:
# EXECUTEI A PARTE ABAIXO EXTRA DADO O SEGUINTE ERRO: erro ImportError: /lib/x86_64-linux-gnu/libp11-kit.so.0: undefined symbol: ffi_type_pointer, version LIBFFI_BASE_7.0

conda update --all #isso esta no venv 
conda install -c conda-forge libffi #isso esta no venv 
conda install -c conda-forge p11-kit #isso esta no venv 
pip uninstall psycopg2 #isso esta no venv 
pip install psycopg2 #isso esta no venv 
export LD_LIBRARY_PATH=$CONDA_PREFIX/lib:$LD_LIBRARY_PATH #isso esta no venv 
source ~/.bashrc #isso esta no venv 
conda activate taiga #isso esta no venv 
pip install django psycopg2 #isso esta no venv 
source activate taiga #isso esta no venv 

#### Passo 4: **Subir o Docker**
Serviços que ficaram dentro do Docker: 
- nginx
- rabbitmq
- postgresql

É necessário entrar na pasta `dockerdb` e digitar o comando para copiar o .env de acordo com o exemplo, seguindo este comando: 
`cp .env.example .env`  

``` 
cd dockerdb/ 
sudo docker compose up -d
```

Após isso, é necessário retornar a pasta root do projeto `taiga-back` com o comando `cd ..` e novamente copiar as váriavéis de ambiente nesse local:
`cp .env.example .env`  


#### Passo 5: **Execução do projeto taiga-back**
A tag `(taiga)`serve apenas para certificar que o comando esta sendo executado dentro da pasta raíz do projeto `taiga-back`.  
  
- Obs: Estas são as configurações inicias do projeto. 

In [ ]:


# EXECUTE O CÓDIGO ACIMA NO CASO DO ERRO LIBFFI_BASE_7.0:
# Execute all migrations to populate the database with basic necessary initial data:
(taiga) DJANGO_SETTINGS_MODULE=settings.config python3 manage.py migrate --noinput
# create an administrator with strong password
(taiga) CELERY_ENABLED=False DJANGO_SETTINGS_MODULE=settings.config python manage.py createsuperuser
#coloque o seu LDAP
user taiga
email nome.sobrenome@exemplo.org
senha taiga

pip install requests --upgrade --upgrade-strategy eager


(taiga) DJANGO_SETTINGS_MODULE=settings.config python manage.py loaddata initial_project_templates
(taiga) DJANGO_SETTINGS_MODULE=settings.config python manage.py compilemessages
(taiga) DJANGO_SETTINGS_MODULE=settings.config python manage.py collectstatic --noinput

In [ ]:
# ADICIONAR UM PROJETO EXEMPLO (OPCIONAL)
CELERY_ENABLED=False DJANGO_SETTINGS_MODULE=settings.config python manage.py sample_data

In [ ]:
# ENFIM RODAR O PROJETO
DJANGO_SETTINGS_MODULE=settings.config python manage.py runserver

In [ ]:
# Caso ocorra mudanças nas migrates é necessário subir estas alterações para o banco de dados com o seguinte comando:
DJANGO_SETTINGS_MODULE=settings.config python manage.py migrate